# 02 · Deduplicate → canonical file (+ annotated raw)

Two-level clustering for multi-recipient awards:
1. **Award level** — connected components over the granular `SOURCE::GRANT_ID` (the key the
   duplicates table uses) → `award_cluster_id` (groups all recipient rows of matched awards).
2. **Recipient level** — within each award cluster, group rows by normalised `OI` and pick the
   canonical source per OI. OIs caught by only one source are kept.

Canonical = highest-priority source. `amount` ← agency copy's native `AMOUNT` + `CURRENCY`;
longest `DESCRIPTION`; other columns fill blanks. Annotates raw with `dedup_status`,
`merged_into`, `enriched`, `award_cluster_id`.

In [ ]:
import pandas as pd, networkx as nx, re
from pathlib import Path

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## CONFIG

In [ ]:
BASE_DIR      = Path('/content/drive/Shareddrives/IOI Team Drive/Research/Projects_Research/Current_Projects_Research/2026_State_of_OI/02_Research/Grant Funding')
RAW_COMBINED  = BASE_DIR / 'raw_combined.csv'
DUP_TABLE     = BASE_DIR / 'duplicates.csv'                        # <-- EDIT filename/path
OUT_DEDUP     = BASE_DIR / 'deduplicated.csv'
OUT_ANNOTATED = BASE_DIR / 'raw_combined_annotated.csv'
DUP_COLS = {'source_a':'SOURCE_A','id_a':'GRANT_ID_A','source_b':'SOURCE_B','id_b':'GRANT_ID_B'}
SOURCE_LABEL_FIXUP = {}

## PARAMETERS

In [ ]:
SOURCE_PRIORITY={'manual':0,'DOD':1,'HHS':1,'NASA':1,'scrapers':2,'openaire':3,'openalex':4}
AGENCY_SOURCES={'DOD','HHS','NASA'}; SPLITTING_SOURCES={'openalex'}
COL_SOURCE='source_dataset'           # harvest tier -> used for canonical PRIORITY
COL_RAW_SOURCE='SOURCE'               # granular provenance -> the duplicates table matches on this
COL_GRANT_ID='GRANT_ID'
COL_AMOUNT='AMOUNT'; COL_CURRENCY='CURRENCY'; COL_DESC='DESCRIPTION'; COL_OI='OI'
# The duplicates table labels openalex rows 'openalex', but raw keeps openalex's UPSTREAM
# provenance in SOURCE ([nsf_award_search], [kaken], ...). Override the match-source for
# such datasets so the keys align. GRANT_IDs are matched case-insensitively (the dedup
# process treated them that way; openalex ids are lowercase in raw, uppercase in the table).
DEDUP_SOURCE_OVERRIDE = {'openalex': 'openalex'}
PROTECTED={'record_uid','source_grantid','dup_key',COL_SOURCE,'dedup_status','merged_into','enriched','award_cluster_id'}

# --- Manual adjudication of over-merge clusters (documented + reproducible) ---
# A handful of awards can't be resolved by rule because the disambiguating fact lives
# OUTSIDE the data (e.g. Wellcome's 'amount awarded' vs 'partnership-inclusive total').
# Each decision is recorded here with its rationale; nothing else is hand-edited.
#
# (1) vireo / DOD: the duplicates table FALSELY links two distinct DoD awards (different
#     years: 2009-10 vs 2011-12) via a truncated-ID match. Drop just that edge so both
#     awards survive separately; the true edge to the $63.7k openalex copy is untouched.
EXCLUDE_DUP_PAIRS = {
    frozenset({'usaspending::w81xwh0920026', 'openalex::w81xwh092002602'}),
}
# (2) Wellcome parent/child: keep the record carrying WELLCOME'S amount-awarded, not the
#     partnership-inclusive total. Force these record_uids to win their (cluster x OI) group.
#       PREreview 214445  -> children reconcile to the parent (£20k+£50k=£70k); keep parent £70k.
#       Europe PMC 108758 -> openaire's £6.6M is partnership-inclusive; Wellcome's own money is
#                            the £1.87M _Z component (per Wellcome's published export); keep _Z.
MANUAL_CANONICAL = {
    'openaire::214445::prereview',
    'scrapers::360g::360G-Wellcome-108758_Z_15_Z::europe pmc',
}

## Load + key dup table

In [ ]:
raw = pd.read_csv(RAW_COMBINED, dtype=str)
if COL_AMOUNT in raw.columns: raw[COL_AMOUNT]=pd.to_numeric(raw[COL_AMOUNT],errors='coerce')
raw = raw.set_index('record_uid', drop=False)
def norm_oi(v): return re.sub(r'\s+',' ',str(v or '').strip().lower())

dups = pd.read_csv(DUP_TABLE, dtype=str)
for c in (DUP_COLS['source_a'],DUP_COLS['source_b']): dups[c]=dups[c].replace(SOURCE_LABEL_FIXUP)
def sg(s,i): return f'{str(s).strip()}::{str(i).strip().lower()}'
dups['sg_a']=[sg(s,i) for s,i in zip(dups[DUP_COLS['source_a']],dups[DUP_COLS['id_a']])]
dups['sg_b']=[sg(s,i) for s,i in zip(dups[DUP_COLS['source_b']],dups[DUP_COLS['id_b']])]
print('Raw rows:', len(raw), '| dup pairs:', len(dups))

Raw rows: 4556 | dup pairs: 1212


## Award-level clustering (on granular SOURCE::GRANT_ID)

The duplicates table matches on the granular `SOURCE` value, so the award key is built from
`SOURCE::GRANT_ID`. Rows with a blank `GRANT_ID` (990-sourced) fall back to their unique
`record_uid` so they stay separate instead of collapsing together.

In [ ]:
gid_str = raw[COL_GRANT_ID].astype(str).str.strip()
gid_blank = raw[COL_GRANT_ID].isna() | gid_str.isin(['','nan','None','NaN'])
match_src = raw[COL_RAW_SOURCE].astype(str).str.strip()
ds = raw[COL_SOURCE].astype(str)
for d_ds, lbl in DEDUP_SOURCE_OVERRIDE.items():
    match_src = match_src.where(ds.ne(d_ds), lbl)
raw['dup_key'] = match_src + '::' + gid_str.str.lower()
raw.loc[gid_blank, 'dup_key'] = raw.loc[gid_blank, 'record_uid']

keys = set(raw['dup_key'])
joinable = int(((dups['sg_a'].isin(keys)) & (dups['sg_b'].isin(keys))).sum())
print(f'Dup pairs joinable (both endpoints in raw): {joinable} / {len(dups)}')
G=nx.Graph(); G.add_nodes_from(keys)
for a,b in zip(dups['sg_a'],dups['sg_b']):
    if a in keys and b in keys and frozenset({a,b}) not in EXCLUDE_DUP_PAIRS:
        G.add_edge(a,b)
k2cid={s:cid for cid,comp in enumerate(nx.connected_components(G)) for s in comp}
raw['award_cluster_id']=raw['dup_key'].map(k2cid)
print('Award clusters:', raw['award_cluster_id'].nunique())

Dup pairs joinable (both endpoints in raw): 1212 / 1212
Award clusters: 3467


## Recipient-level dedup (per award cluster × OI)

In [ ]:
def rank(u): return SOURCE_PRIORITY.get(raw.at[u,COL_SOURCE],99)
def blank(v): return pd.isna(v) or (isinstance(v,str) and v.strip()=='')
FILL=[c for c in raw.columns if c not in PROTECTED]
raw['_grp']=raw['award_cluster_id'].astype(str)+'||'+raw[COL_OI].map(norm_oi)

records,status={},{}
for _,idx in raw.groupby('_grp').groups.items():
    members=sorted(list(idx), key=rank)
    forced=[m for m in members if m in MANUAL_CANONICAL]
    if forced:
        can=forced[0]; members=[can]+[m for m in members if m!=can]
    else:
        can=members[0]
    row=raw.loc[can].to_dict(); enr=False
    if len(members)>1:
        ag=[m for m in members if raw.at[m,COL_SOURCE] in AGENCY_SOURCES]
        if ag and COL_AMOUNT in row:
            a=raw.at[ag[0],COL_AMOUNT]
            if not blank(a) and a!=row.get(COL_AMOUNT):
                row[COL_AMOUNT]=a
                if COL_CURRENCY in row: row[COL_CURRENCY]=raw.at[ag[0],COL_CURRENCY]
                enr=True
        if COL_DESC in row:
            best,bl=row.get(COL_DESC),len(str(row.get(COL_DESC) or ''))
            for m in members:
                d=raw.at[m,COL_DESC]
                if not blank(d) and len(str(d))>bl: best,bl=d,len(str(d))
            if best!=row.get(COL_DESC): row[COL_DESC]=best; enr=True
        for f in FILL:
            if f in (COL_AMOUNT,COL_DESC,COL_CURRENCY,'_grp'): continue
            if blank(row.get(f)):
                for m in members[1:]:
                    v=raw.at[m,f]
                    if not blank(v): row[f]=v; enr=True; break
    flagged=(raw.at[can,COL_SOURCE] in SPLITTING_SOURCES) and len(members)>1
    row.pop('_grp', None)
    records[can]=row
    status[can]={'dedup_status':'kept_flagged' if flagged else 'kept','merged_into':pd.NA,'enriched':enr}
    for m in members[1:]: status[m]={'dedup_status':'dropped','merged_into':can,'enriched':False}

## Write + summary

In [ ]:
raw=raw.drop(columns=['_grp'])
raw['dedup_status']=[status[u]['dedup_status'] for u in raw['record_uid']]
raw['merged_into']=[status[u]['merged_into'] for u in raw['record_uid']]
raw['enriched']=[status[u]['enriched'] for u in raw['record_uid']]
dedup=pd.DataFrame([records[c] for c in records])
dedup['dedup_status']=[status[c]['dedup_status'] for c in records]
dedup=dedup.drop(columns=['dup_key'], errors='ignore'); raw=raw.drop(columns=['dup_key'], errors='ignore')
dedup.to_csv(OUT_DEDUP,index=False); raw.to_csv(OUT_ANNOTATED,index=False)
print(f'Raw {len(raw):,} -> deduplicated {len(dedup):,} | dropped {int((raw.dedup_status=="dropped").sum()):,}'
      f' | enriched {int(raw.enriched.sum()):,} | flagged {int((raw.dedup_status=="kept_flagged").sum()):,}')

Raw 4,556 -> deduplicated 3,560 | dropped 996 | enriched 694 | flagged 0


In [ ]:
#diagnostics - understand same source duplicate clusters
ann = pd.read_csv(OUT_ANNOTATED, dtype=str)
ann['AMOUNT_n'] = pd.to_numeric(ann['AMOUNT'], errors='coerce')

PRIMARY = {'manual','DOD','HHS','NASA','scrapers'}
prim = ann[ann.source_dataset.isin(PRIMARY)].copy()
prim['akey'] = prim.SOURCE.str.strip() + '::' + prim.GRANT_ID.astype(str).str.strip().str.lower()

nkeys = prim.groupby('award_cluster_id')['akey'].nunique()
over = nkeys[nkeys >= 2].index.tolist()
print('Over-merged clusters:', len(over), '->', over)

cols = ['source_dataset','GRANT_ID','OI','AMOUNT_n','START_DATE','END_DATE']
for cid in over:
    sub = ann[ann.award_cluster_id == cid]
    print('\ncluster', cid)
    print(sub.sort_values('source_dataset')[cols].to_string(index=False))

Over-merged clusters: 2 -> ['242', '29']

cluster 242
source_dataset                          GRANT_ID        OI  AMOUNT_n START_DATE   END_DATE
      openaire                            214445 prereview   70000.0 2019-02-18 2020-03-31
      openalex       360G-Wellcome-214445_Z_18_A prereview   20000.0 2019-07-31 2021-04-29
      openalex       360G-Wellcome-214445_Z_18_Z prereview   50000.0 2019-02-18 2020-10-17
      scrapers 360g::360G-Wellcome-214445_Z_18_A prereview   20000.0 2019-07-31 2021-04-29
      scrapers 360g::360G-Wellcome-214445_Z_18_Z prereview   50000.0 2019-02-18 2020-10-17

cluster 29
source_dataset                          GRANT_ID         OI   AMOUNT_n START_DATE   END_DATE
      openaire                            108758 europe pmc 6615780.00 2016-04-01 2021-03-31
      openalex       360G-Wellcome-108758_Z_15_C europe pmc       0.00 2018-06-04 2021-04-03
      openalex       360G-Wellcome-108758_Z_15_Z europe pmc 1874248.09 2016-04-01 2021-03-31
      openalex  